In [2]:
import pandas as pd
df_productos_gamer_argentina = pd.read_csv('productos_gamer_argentina.csv')

Si el filtrado te ayuda a limpiar el "pantano", **`groupby`** es el motor de tu fábrica. Es la herramienta principal en la etapa de Transformación de cualquier pipeline de Data Engineering.

Para dominarlo, debes entender el concepto mental detrás de esta función, conocido como **"Split-Apply-Combine"** (Separar - Aplicar - Combinar).

1. **Split (Separar):** Pandas rompe tu DataFrame gigante en mini-DataFrames, agrupando las filas que comparten el mismo valor en una columna (ej: agrupa todos los "Ryzen 5" por un lado y los "Core i5" por otro).
2. **Apply (Aplicar):** Le aplicas una función matemática a cada mini-DataFrame por separado (ej: calcular el `mean()` o promedio de precio).
3. **Combine (Combinar):** Pandas vuelve a unir todos los resultados en un nuevo DataFrame limpio y resumido.

---

### 🧠 ¿Cuándo usar `groupby`?

* **En ETLs:** Para comprimir datos granulares antes de enviarlos a una base de datos (ej: tienes un millón de ventas diarias, pero a tu PostgreSQL solo le envías el total agrupado por mes).
* **En Análisis Exploratorio:** Para responder preguntas de negocio rápidas ("¿Qué tienda tiene el precio promedio más bajo para esta tarjeta gráfica?").
* **Para crear métricas calculadas:** Generar KPIs como el Ticket Promedio o el Valor Bruto de Mercancía (GMV) agrupado por categoría.

---

### ⚙️ Parámetros Clave (Los que importan de verdad)

| Parámetro | Opciones / Uso | ¿Para qué sirve? |
| --- | --- | --- |
| `by` | String o Lista de Strings | La columna (o columnas) por las que vas a separar los datos. Ej: `by='tienda'` o `by=['tienda', 'marca']`. |
| **`as_index`** | `True` (default), **`False`** | **El truco de los pros.** Si lo dejas en `True`, la columna agrupada se vuelve el Índice (molesto para exportar a SQL). Usa `False` para mantener un formato de tabla clásico y plano. |
| `dropna` | `True` (default), `False` | Si es `True`, ignora las filas donde la columna de agrupación tiene `NaN`. Usa `False` si quieres ver un grupo llamado `NaN` (útil para auditar qué datos te faltan). |

---

### 💻 Ejemplos Prácticos por Contexto

#### 1. Contexto: Web Scraping de Hardware (Tu ArgHardware Tracker)

Imagina que los spiders de tu proyecto acaban de raspar miles de precios de diferentes páginas, y necesitas calcular el precio promedio real de cada componente en el mercado argentino de hoy.

In [4]:
# 1. Agrupación Básica (Un solo grupo, un solo cálculo)
# Calculamos el precio promedio de cada componente.
# OJO: Usamos as_index=False para que 'componente' siga siendo una columna normal.
df_promedios = df_productos_gamer_argentina.groupby(by='producto', as_index=False)['precio'].mean()

df_promedios.head()

,producto,precio
0,Fuente EVGA 750W Gold,185000.0
1,Gabinete Corsair 4000D Airflow,145000.0
2,Gabinete Lian Li O11 Dynamic,220000.0
3,GeForce RTX 4060 Ti,620000.0
4,GeForce RTX 4070 Super,1250000.0


In [6]:
# 2. Agrupación Múltiple (Lista de columnas)
# ¿Cuál es el precio promedio de cada componente, PERO separado por tienda?
df_promedios_tienda = df_productos_gamer_argentina.groupby(
    by=['producto', 'tienda'], 
    as_index=False
)['precio'].mean()

df_promedios_tienda.head()

,producto,tienda,precio
0,Fuente EVGA 750W Gold,Venex,185000.0
1,Gabinete Corsair 4000D Airflow,Gezatek,145000.0
2,Gabinete Lian Li O11 Dynamic,Gezatek,220000.0
3,GeForce RTX 4060 Ti,CompraGamer,620000.0
4,GeForce RTX 4070 Super,CompraGamer,1250000.0


#### 2. Contar operaciones: Usamos .count() o .size()
- .size() cuenta todas las filas del grupo. 
- .count() cuenta los valores no nulos.


In [10]:
# Cuanto procesadores hay
df_procesadores = df_productos_gamer_argentina \
    .query('categoria == "Procesador" ') \
    .groupby(by='categoria', as_index=False)['producto'].size()

df_procesadores

,categoria,size
0,Procesador,20


In [12]:
# Sumar precios:
df_ingresos = df_productos_gamer_argentina \
    .query('categoria == "Procesador" ') \
    .groupby(by='categoria', as_index=False)['precio'].sum()
    
df_ingresos

,categoria,precio
0,Procesador,10100000.0


#### 3. 🚀 El Nivel Dios: `.agg()` (Múltiples cálculos a la vez)

Hacer un `groupby` por cada métrica es ineficiente. Los ingenieros de datos usan el método `.agg()` (Aggregate) pasándole un diccionario. Te permite calcular la suma de una columna y el promedio de otra en una sola pasada.




In [16]:
# Calculamos el GMV (Total facturado) y el AOV (Ticket Promedio) en un solo bloque
df_kpis = df_productos_gamer_argentina \
    .groupby('categoria', as_index=False) \
        .agg(
            cantidad=('producto', "count"),
            precio_promedio=('precio', 'mean'), 
            precio_max=('precio', 'max'),
            precio_min=('precio', 'min')
        )

print(df_kpis)

         categoria  cantidad  precio_promedio  precio_max  precio_min
0   Almacenamiento         5         135000.0    135000.0    135000.0
1          Fuentes         5         185000.0    185000.0    185000.0
2         Gabinete        10         182500.0    220000.0    145000.0
3         Memorias         5         180000.0    180000.0    180000.0
4        Monitores        10         510000.0    580000.0    440000.0
5      Motherboard         5         290000.0    290000.0    290000.0
6      Periféricos        15          85000.0    115000.0     45000.0
7  Placas de Video        20         835000.0   1250000.0    490000.0
8       Procesador        20         505000.0    950000.0    310000.0
9    Refrigeración         5         155000.0    155000.0    155000.0


*Nota: Fíjate cómo al usar tuplas dentro de `.agg()`, le asignas automáticamente el nombre final a la columna resultante (`cantidad`, `precio_promedio`, etc...). Tu dataset sale listo para insertar en una base de datos o conectar a Power BI.*

---

### ⚠️ El Error más común del Junior

Nunca intentes hacer un `groupby` sobre un DataFrame entero sin especificar qué columna numérica quieres calcular al final.

* ❌ **MAL:** `df.groupby('tienda').mean()` -> Pandas intentará calcular el promedio de *todas* las columnas, incluyendo los nombres de los productos o fechas, y lanzará un error o consumirá toda tu memoria.
* ✅ **BIEN:** `df.groupby('tienda')['precio'].mean()` -> Primero filtras la columna numérica que te interesa, luego aplicas el cálculo.
